In [69]:
import numpy as np
import os #allows python to interact with the operating system
os.environ["KERAS_BACKEND"] = "tensorflow" # Tell Keras to use TF as its engine
import keras
# source Gradioexp/bin/activate  for terminal activation

MNIST is a classic dataset of 70,000 handwritten digit images (0–9), each being a 28×28 grayscale image.

# Data Loading & Preprocessing

In [70]:
#1) Load the data and split it between train and test sets

#keras.datasets.mnist.load_data():Built-in Keras utility function that automatically downloads, caches, and loads the MNIST dataset directly from the internet — no manual downloading needed.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

#70,000 total
#60,000 → Training (85.7%)
#10,000 → Testing  (14.3%)

#2) Normalize (Scale images to the [0, 1] range)
 
# Raw pixel values are integers from 0 to 255
# Dividing by 255 squishes them to 0.0 to 1.0

#Neural networks train faster and more stably with small input values.
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

#3) Reshape to make data ready to feed into a CNN
#CNNs (Convolutional Neural Networks) expect input as (batch, height, width, channels)
# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape) #one-dimensional array
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


x_train shape: (60000, 28, 28, 1)
y_train shape: (60000,)
60000 train samples
10000 test samples


In [71]:
# Model parameters
num_classes = 10
input_shape = (28, 28, 1)

What is keras.Sequential?
It's a linear stack of layers where data flows from top to bottom, one layer at a time.

In [72]:
model = keras.Sequential(
    [
        keras.layers.Input(shape=input_shape), #(28, 28, 1)
        keras.layers.Flatten(), #Unrolls the 2D image into a 1D vector (28*28*1 = 784)
        keras.layers.Dense(256, activation="relu"), #learn features
        #256 neurons, each connected to all 784 inputs
        #784 inputs × 256 neurons = 200,704 connections
        keras.layers.Dropout(0.2),#randomly switch OFF 20% of neurons during training to prevent overfitting
        keras.layers.Dense(128, activation="relu"),#Learns higher-level combinations of patterns from layer above
        keras.layers.Dropout(0.2),
        keras.layers.Dense(num_classes, activation="softmax"),#softmax converts raw scores into probabilities that sum to 1.0

    ]
)

# Neural Network Architecture
```
   Input Image (28×28×1)
        │
        ▼
   Flatten → 784 values
        │
        ▼
   Dense(256) + ReLU → 256 feature signals
        │
        ▼
   Dropout(0.2) → ~205 active neurons
        │
        ▼
   Dense(128) + ReLU → 128 abstract features
        │
        ▼
   Dropout(0.2) → ~102 active neurons
        │
        ▼
   Dense(10) + Softmax → [0.01, 0.02, 0.91, ...]
                                        ▲
                                  Final prediction
```


In [73]:
# model = keras.Sequential(
#     [
#         keras.layers.Input(shape=input_shape),
#         keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
#         keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
#         keras.layers.MaxPooling2D(pool_size=(2, 2)),
#         keras.layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
#         keras.layers.Conv2D(128, kernel_size=(3, 3), activation="relu"),
#         keras.layers.GlobalAveragePooling2D(),
#         keras.layers.Dropout(0.5),
#         keras.layers.Dense(num_classes, activation="softmax"),
#     ]
# )


In [74]:
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

```
(input_features × neurons) + biases
(784 × 256) + 256
= 200,704 + 256
= 200,960 
```

In [75]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(), 
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="acc"),
    ],
)

#updates the weights to reduce the error (loss)
#Adam = Adaptive Moment Estimation(each weight has its own learning rate that adapts during training)


In [76]:
batch_size = 128
epochs = 20

callbacks = [
    #keras.callbacks.ModelCheckpoint(filepath="model_at_epoch_{epoch}.keras"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2),
]

model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.15,
    callbacks=callbacks,
)
score = model.evaluate(x_test, y_test, verbose=0)


Epoch 1/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - acc: 0.8867 - loss: 0.3802 - val_acc: 0.9614 - val_loss: 0.1329
Epoch 2/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - acc: 0.9530 - loss: 0.1561 - val_acc: 0.9691 - val_loss: 0.1043
Epoch 3/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - acc: 0.9670 - loss: 0.1100 - val_acc: 0.9757 - val_loss: 0.0808
Epoch 4/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - acc: 0.9728 - loss: 0.0886 - val_acc: 0.9770 - val_loss: 0.0793
Epoch 5/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - acc: 0.9765 - loss: 0.0733 - val_acc: 0.9766 - val_loss: 0.0764
Epoch 6/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - acc: 0.9814 - loss: 0.0602 - val_acc: 0.9772 - val_loss: 0.0718
Epoch 7/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - acc: 0.9830 - loss: 0.0539 - val_acc: 0.9782 - val_loss: 0.0754
Epoch 8/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - acc: 0.9852 - loss: 0.0471 - val_acc: 0.9798 - val_loss: 0.0706
Epoch 9/20
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - a

In [77]:
# log_dir = "logs"

# tensorboard_callback = keras.callbacks.TensorBoard(
#     log_dir=log_dir,
#     histogram_freq=1
# )

In [78]:
# model.fit(
#     x_train,
#     y_train,
#     batch_size=128,
#     epochs=20,
#     validation_split=0.15,
#     callbacks=[
#         keras.callbacks.EarlyStopping(monitor="val_loss", patience=2),
#         tensorboard_callback
#     ]
# )

In [79]:
model.save("final_model.keras")


In [80]:
model = keras.saving.load_model("final_model.keras")


In [81]:
predictions = model.predict(x_test)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [82]:
y_pred = np.argmax(predictions, axis=1)

In [83]:
y_test

array([7, 2, 1, ..., 4, 5, 6], shape=(10000,), dtype=uint8)

In [84]:
y_pred

array([7, 2, 1, ..., 4, 5, 6], shape=(10000,))

In [85]:
accuracy = np.mean(y_pred == y_test)
print(f"Accuracy: {accuracy*100.:.4f}")

Accuracy: 98.0000


In [86]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred, digits=4))
cm = confusion_matrix(y_test, y_pred)

              precision    recall  f1-score   support

           0     0.9779    0.9918    0.9848       980
           1     0.9938    0.9815    0.9876      1135
           2     0.9778    0.9816    0.9797      1032
           3     0.9708    0.9861    0.9784      1010
           4     0.9787    0.9807    0.9797       982
           5     0.9852    0.9731    0.9791       892
           6     0.9843    0.9823    0.9833       958
           7     0.9786    0.9805    0.9796      1028
           8     0.9792    0.9671    0.9731       974
           9     0.9733    0.9742    0.9737      1009

    accuracy                         0.9800     10000
   macro avg     0.9800    0.9799    0.9799     10000
weighted avg     0.9800    0.9800    0.9800     10000



In [87]:
print(cm)


[[ 972    1    1    1    0    2    1    0    1    1]
 [   0 1114    5    1    0    1    2    2   10    0]
 [   6    0 1013    3    1    0    2    4    3    0]
 [   1    0    4  996    0    1    0    5    3    0]
 [   1    0    2    1  963    0    3    2    0   10]
 [   2    0    0   10    2  868    4    1    2    3]
 [   6    2    0    2    4    3  941    0    0    0]
 [   1    2    7    2    1    0    0 1008    0    7]
 [   2    0    4    7    3    5    1    4  942    6]
 [   3    2    0    3   10    1    2    4    1  983]]


In [89]:
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Prepare Data (using the MNIST setup from before)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# 2. Define a simple model
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 3. SETUP TENSORBOARD
# Create a directory for logs with a timestamp
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# Define the TensorBoard callback
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# 4. Train the model with the callback
print("Training model... Check your terminal to launch TensorBoard.")
model.fit(x_train, y_train, 
          epochs=5, 
          validation_data=(x_test, y_test), 
          callbacks=[tensorboard_callback])

c:\Users\hp\anaconda3\envs\ai\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training model... Check your terminal to launch TensorBoard.
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9134 - loss: 0.2952 - val_accuracy: 0.9577 - val_loss: 0.1409
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 14s 7ms/step - accuracy: 0.9574 - loss: 0.1409 - val_accuracy: 0.9687 - val_loss: 0.1049
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9674 - loss: 0.1054 - val_accuracy: 0.9736 - val_loss: 0.0854
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9723 - loss: 0.0871 - val_accuracy: 0.9754 - val_loss: 0.0789
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9772 - loss: 0.0730 - val_accuracy: 0.9783 - val_loss: 0.0714


In [ ]:
# tensorboard --logdir logs/fit